In [31]:
# pip install requests
# !pip install psycopg2
import requests
import pandas as pd
import json
import time
from datetime import datetime, timedelta
import psycopg2

In [57]:
# connect dataframe with AWS RDS database

endpoint = "ta43-onboarding.c5kcsm8im4cz.ap-southeast-2.rds.amazonaws.com"
master_username = "postgres"
password = "TA43Onboarding"
database = "postgres"
# port = 5432

conn_db = psycopg2.connect(host=endpoint, dbname=database, user=master_username, password=password)
cur = conn_db.cursor()

print("Connect to postgres")

Connect to postgres


In [58]:
# check connection

cur.execute("SELECT NOW();")
print(cur.fetchone()[0])  # prints current timestamp

2025-08-09 10:26:19.027307+00:00


In [34]:
# City of Melbourne token
headers = {
    "key_com" : "8272f3e3cf855ca3006bd9d38e135f06a0714adb8519bf503af526af"
}

In [35]:
# # Real-time: On-street parking bay sensors

# # On-street parking bay sensors
# onstreet_pbs_url = "https://data.melbourne.vic.gov.au/api/explore/v2.1/catalog/datasets/on-street-parking-bay-sensors/records"

# headers = {
#     "key_com" : "8272f3e3cf855ca3006bd9d38e135f06a0714adb8519bf503af526af"
# }
# # record limit per iteration
# record_limit = 100


# # loop interval: 2 min.
# loop_interval = 10

# # initialise current time - buffer
# current_time_buffer = (datetime.now() - timedelta(seconds=20)).isoformat()


# while True:
#     # initialise list for storing updated_data
#     updated_lst = []
#     offset = 0
#     while True:
#         condition = {"limit" : record_limit,
#                     "offset" : offset,
#                     "where" : f"lastupdated > '{current_time_buffer}'"}
        
#         # Get request
#         onstreet_pbs_response = requests.get(onstreet_pbs_url,
#                                             headers=headers,
#                                             params=condition)
#         # print test
#         test = onstreet_pbs_response.json()
#         print(test)

#         # HTTP breakout condition
#         res_status_cd = onstreet_pbs_response.status_code
#         if res_status_cd != 200:
#             if res_status_cd >= 500:
#                 print(f"Failed: {res_status_cd} Server Error")
#             else:
#                 print(f"Failed: {res_status_cd} Request Error", onstreet_pbs_response.text)
#             break

#         # Get response into record, csv format
#         record_json = onstreet_pbs_response.json()
#         record_onstreet_pbs = record_json.get("results", [])
#         print(f"Offset {offset}, fetched {len(record_onstreet_pbs)}")

#         # break condition if no record 
#         if not record_onstreet_pbs:
#             break
        
#         # append new record into list
#         updated_lst.extend(record_onstreet_pbs)

#         # update offest with record_limit
#         offset += record_limit

#     if updated_lst:
#         # convert json into pandas dataframe for data mapping
#         df_onstreet_pbs = pd.DataFrame(updated_lst)
#         print(f"{len(df_onstreet_pbs)} records updated.")

#         # update current time
#         current_time_buffer = max(pd.to_datetime(df_onstreet_pbs["lastupdated"]))
    
#     else:
#         print("No record updates.")
    
#     # loop report
#     time.sleep(loop_interval)

# print(df_onstreet_pbs)

In [36]:
# 1 (Get 100 records) On-street parking bay sensors
onstreet_pbs_url = "https://data.melbourne.vic.gov.au/api/explore/v2.1/catalog/datasets/on-street-parking-bay-sensors/records?limit=100"

# Get request
onstreet_pbs_response = requests.get(onstreet_pbs_url, headers=headers)

# Output
onstreet_pbs_data = onstreet_pbs_response.json()

In [73]:
# extract record list
record_onstreet_pbs = onstreet_pbs_data["results"]

# convert json to dataframe
df_onstreet_pbs = pd.json_normalize(record_onstreet_pbs)

# rename column header
df_onstreet_pbs = df_onstreet_pbs.rename(
    columns={"zone_number" : "parkingzone",
             "status_description" : "status_desc", 
             "location.lon" : "longitude",
             "location.lat" : "latitude"}
    )

# convert datetime columns
df_onstreet_pbs["lastupdated"] = pd.to_datetime(df_onstreet_pbs["lastupdated"])
# df_onstreet_pbs["lastupdated"] = df_onstreet_pbs["lastupdated"].dt.strftime("%H:%M:%S")

df_onstreet_pbs["status_timestamp"] = pd.to_datetime(df_onstreet_pbs["status_timestamp"])
# df_onstreet_pbs["status_timestamp"] = df_onstreet_pbs["status_timestamp"].dt.strftime("%H:%M:%S")
df_onstreet_pbs["parking_date"] = df_onstreet_pbs["status_timestamp"].dt.date
df_onstreet_pbs["parking_time"] = df_onstreet_pbs["status_timestamp"].dt.time

df_onstreet_pbs["parkingzone"] = df_onstreet_pbs["parkingzone"].astype("Int64")


def parking_status(x):
    """
    Function convert parking status.
    Parameter: x is parking bay sensor record from status_description
    Returns:
    1 if parking is available (Unoccupied),
    0 if parking is not available (Present)
    """
    if x["status_desc"] == "Unoccupied":
        return 1
    else:
        return 0

df_onstreet_pbs["is_available"] = df_onstreet_pbs.apply(parking_status, axis=1)

# output
# print(df_onstreet_pbs.head())
# print(df_onstreet_pbs.dtypes)


# For SQL create table: PARKING_BAY_SENSOR (pk = kerbsideid, fk = parkingzone)
tb_parking_bay_sensor = df_onstreet_pbs[["kerbsideid", "lastupdated", "parking_date", "parking_time", "is_available", "parkingzone"]]

In [38]:
# # 2. Sign plates located in each parking zone
# sign_plates_loc_url = "https://data.melbourne.vic.gov.au/api/explore/v2.1/catalog/datasets/sign-plates-located-in-each-parking-zone/records?limit=100"

# # Get request
# sign_plates_loc_response = requests.get(sign_plates_loc_url, headers=headers)

# # Output
# sign_plates_loc_data = sign_plates_loc_response.json()

In [39]:
# # extract record list
# record_sign_plates_loc = sign_plates_loc_data["results"]

# # convert json to dataframe
# df_sign_plates_loc = pd.json_normalize(record_sign_plates_loc)

# # convert datetime columns
# df_sign_plates_loc["time_restrictions_start"] = pd.to_datetime(df_sign_plates_loc["time_restrictions_start"])
# df_sign_plates_loc["time_restrictions_start"] = df_sign_plates_loc["time_restrictions_start"].dt.time

# df_sign_plates_loc["time_restrictions_finish"] = pd.to_datetime(df_sign_plates_loc["time_restrictions_finish"])
# df_sign_plates_loc["time_restrictions_finish"] = df_sign_plates_loc["time_restrictions_finish"].dt.time

# # for checking
# print(df_sign_plates_loc)

# print(df_sign_plates_loc["restriction_display"].unique())

# # # Day of week dict {day:idx}
# # day_dict = {
# #     ""
# # }

In [ ]:
# 2. (Static record) Sign plates located in each parking zone
with open("Dataset/sign-plates-located-in-each-parking-zone.json", "r") as file:
    data_sign_plates = json.load(file)

# convert json to dataframe
df_sign_plates = pd.json_normalize(data_sign_plates)

# format column data type
df_sign_plates["time_restrictions_start"] = pd.to_datetime(df_sign_plates["time_restrictions_start"], format="%H:%M:%S")

df_sign_plates["time_restrictions_finish"] = pd.to_datetime(df_sign_plates["time_restrictions_finish"], format="%H:%M:%S")

# output
print(df_sign_plates.head())
print(df_sign_plates.dtypes)

# For SQL create table: SIGN_PLATE (pk = parkingzone)
tb_sign_plates = df_sign_plates

   parkingzone restriction_days time_restrictions_start  \
0         7033          Mon-Fri     1900-01-01 07:30:00   
1         7047              Sat     1900-01-01 07:30:00   
2         7068              Sat     1900-01-01 07:30:00   
3         7089              Sat     1900-01-01 07:30:00   
4         7109              Sat     1900-01-01 07:30:00   

  time_restrictions_finish restriction_display  
0      1900-01-01 18:30:00                  2P  
1      1900-01-01 12:30:00                  2P  
2      1900-01-01 12:30:00                  1P  
3      1900-01-01 12:30:00                  2P  
4      1900-01-01 12:30:00                  1P  
parkingzone                          int64
restriction_days                    object
time_restrictions_start     datetime64[ns]
time_restrictions_finish    datetime64[ns]
restriction_display                 object
dtype: object


In [41]:
# count_dist= df_sign_plates["parkingzone"].unique()
# pd.value_counts(count_dist)

In [42]:
# # 3. Parking zone linked to street segments
# parking_zone_url = "https://data.melbourne.vic.gov.au/api/explore/v2.1/catalog/datasets/parking-zones-linked-to-street-segments/records?limit=100"

# # Get request
# parking_zone_response = requests.get(parking_zone_url, headers=headers)

# # Output
# parking_zone_data = parking_zone_response.json()

In [43]:
# # extract record list
# record_parking_zone = parking_zone_data["results"]

# # convert json to dataframe
# record_parking_zone = pd.json_normalize(record_parking_zone)

# print(record_parking_zone)
# print(record_parking_zone.dtypes)

In [ ]:
# 3. (Static record) Parking zone linked to street segments
with open("Dataset/parking-zones-linked-to-street-segments.json", "r") as file:
    data_parking_zones = json.load(file)

df_parking_zones = pd.json_normalize(data_parking_zones)
print(df_parking_zones)

# For SQL create table: PARKING_ZONE (pk = parkingzone, fk = segment_id)
tb_parking_zones = df_parking_zones

     parkingzone         onstreet            streetfrom            streetto  \
0           7000      Poplar Road       Upfield Railway      Kendall Avenue   
1           7031  Cardigan Street    Argyle Place North      Grattan Street   
2           7068     Lygon Street        Faraday Street        Elgin Street   
3           7067     Lygon Street         Pelham Street  Argyle Place North   
4           7118  Swanston Street  Lincoln Square North      Grattan Street   
..           ...              ...                   ...                 ...   
793         7963   Rosslyn Street         Howard Street         King Street   
794         7950    Railway Place        Stanley Street        Roden Street   
795         7968   Spencer Street       La Trobe Street     Jeffcott Street   
796         7975   Stanley Street           King Street      Spencer Street   
797         7995   William Street          Capel Street      Rosslyn Street   

     segment_id  
0         22405  
1         20512

In [ ]:
# (static) 4. On-street parking bays
with open("Dataset/on-street-parking-bays.json", "r") as file:
    data_parking_bays = json.load(file)

# convert json to dataframe
df_parking_bays = pd.json_normalize(data_parking_bays)

# rename column
df_parking_bays = df_parking_bays.rename(columns={"roadsegmentid" : "segment_id",
                                                  "roadsegmentdescription" : "segment_desc"})

# format column data type
df_parking_bays["kerbsideid"] = pd.to_numeric(df_parking_bays["kerbsideid"], errors="coerce")

df_parking_bays["lastupdated"] = pd.to_datetime(df_parking_bays["lastupdated"], format="%Y-%m-%d")

# filter necessary column
df_parking_bays = df_parking_bays.drop(columns=["location.lon","location.lat"])


# print(df_parking_bays.head())
# print(df_parking_bays.dtypes)

# For SQL create table: PARKING_BAY (pk = kerbsideid, fk = segment_id)
tb_parking_bays = df_parking_bays[["kerbsideid", "segment_id", "segment_desc", "latitude", "longitude"]]

In [ ]:
# this merged only work when their is enough record available
# df_static_merged = (
#     df_parking_bays.merge(df_parking_zones, on="segment_id", how="right")
#     .merge(df_sign_plates, on="parkingzone", how="right")
# )

# print(df_static_merged)

       segment_id  kerbsideid  \
0         20508.0         NaN   
1         20508.0         NaN   
2         20508.0         NaN   
3         20508.0         NaN   
4         20508.0         NaN   
...           ...         ...   
54992         NaN         NaN   
54993         NaN         NaN   
54994         NaN         NaN   
54995         NaN         NaN   
54996         NaN         NaN   

                                  roadsegmentdescription   latitude  \
0      Cardigan Street between Argyle Place South and... -37.803143   
1      Cardigan Street between Argyle Place South and... -37.803119   
2      Cardigan Street between Argyle Place South and... -37.803176   
3      Cardigan Street between Argyle Place South and... -37.802873   
4      Cardigan Street between Argyle Place South and... -37.802926   
...                                                  ...        ...   
54992                                                NaN        NaN   
54993                              

In [56]:
df_static_merged.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 54997 entries, 0 to 54996
Data columns (total 14 columns):
 #   Column                    Non-Null Count  Dtype         
---  ------                    --------------  -----         
 0   segment_id                54593 non-null  float64       
 1   kerbsideid                25697 non-null  float64       
 2   roadsegmentdescription    54582 non-null  object        
 3   latitude                  54582 non-null  float64       
 4   longitude                 54582 non-null  float64       
 5   lastupdated               54582 non-null  datetime64[ns]
 6   parkingzone               54997 non-null  int64         
 7   onstreet                  54593 non-null  object        
 8   streetfrom                54593 non-null  object        
 9   streetto                  54490 non-null  object        
 10  restriction_days          54997 non-null  object        
 11  time_restrictions_start   54997 non-null  datetime64[ns]
 12  time_restrictions_

In [47]:
# # this merge only contain 8 record
# parking_merge1 = pd.merge(
#     df_onstreet_pbs,
#     record_parking_zone,
#     left_on=["parkingzone"],
#     right_on=["parkingzone"],
#     how="inner"
# )

# print(parking_merge1)

In [48]:
# create engine for connection
from sqlalchemy import create_engine, text

endpoint = "ta43-onboarding.c5kcsm8im4cz.ap-southeast-2.rds.amazonaws.com"
database = "postgres"  # or your actual DB
username = "postgres"
password = "TA43Onboarding"

rds_engine = create_engine(
    f"postgresql+psycopg2://{username}:{password}@{endpoint}:5432/{database}?sslmode=require"
)

# check connection
with rds_engine.connect() as conn:
    print(conn.execute(text("SELECT now()")).scalar())

2025-08-09 08:15:56.244658+00:00


In [ ]:
# export dataframe from vs code to aws
# df_onstreet_pbs.to_sql("parking_bay_sensors", rds_engine, if_exists="append", index=False, chunksize=10_000, method="multi")
# df_static_merged.to_sql("static_parking_merged", rds_engine, if_exists="append", index=False, chunksize=10_000, method="multi")

In [ ]:
# export dataframe from vs code to aws

# REAL-TIME DB
# create table: PARKING_BAY_SENSOR (pk = kerbsideid, fk = parkingzone)
tb_parking_bay_sensor.to_sql("parking_bay_sensor", rds_engine, if_exists="append", index=False, chunksize=10_000, method="multi")

# STATIC DB
# For SQL create table: SIGN_PLATE (pk = parkingzone)
tb_sign_plates.to_sql("sign_plate", rds_engine, if_exists="append", index=False, chunksize=10_000, method="multi")

# For SQL create table: PARKING_ZONE (pk = parkingzone, fk = segment_id)
tb_parking_zones.to_sql("parking_zone", rds_engine, if_exists="append", index=False, chunksize=10_000, method="multi")

# For SQL create table: PARKING_BAY (pk = kerbsideid, fk = segment_id)
tb_parking_bays.to_sql("parking_bay", rds_engine, if_exists="append", index=False, chunksize=10_000, method="multi")

In [79]:
tb_parking_bay_sensor.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 6 columns):
 #   Column        Non-Null Count  Dtype              
---  ------        --------------  -----              
 0   kerbsideid    100 non-null    int64              
 1   lastupdated   100 non-null    datetime64[ns, UTC]
 2   parking_date  100 non-null    object             
 3   parking_time  100 non-null    object             
 4   is_available  100 non-null    int64              
 5   parkingzone   90 non-null     Int64              
dtypes: Int64(1), datetime64[ns, UTC](1), int64(2), object(2)
memory usage: 4.9+ KB
